# Final Project — Data Science in Cybersecurity
## Critical Evaluation & Reproduction of: Phishing Website Detection

**Source Repository:** [sujeetgund/phishing-website-detection](https://github.com/sujeetgund/phishing-website-detection)  
**Dataset:** [UCI Phishing Websites Data Set](https://archive.ics.uci.edu/dataset/327/phishing+websites) (11,055 rows, 30 features)  
**Problem:** Binary classification of websites as Phishing (1) or Legitimate (-1) using URL and metadata features.

---

### Notebook Structure
1. **Source Reproduction** — Faithful reproduction of the author's 5-model comparison
2. **Thorough EDA** — 15+ data quality checks with visualizations
3. **Feature Engineering Experiments** — Importance, mutual information, RFE, PCA
4. **Model Training & Comparison** — 5 models with proper train/val/test split
5. **Full Evaluation Metrics** — Precision, Recall, F1, F2, MCC, ROC-AUC, PR-AUC
6. **Threshold Selection** — Optimal threshold tuning on validation set
7. **Error Analysis** — Detailed FP/FN inspection with cybersecurity implications

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import RFE, mutual_info_classif, SelectKBest
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, matthews_corrcoef,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)

np.random.seed(42)
print("All imports successful.")

## 1. Data Loading & Inspection

We load the UCI Phishing Websites dataset. The dataset contains 30 features extracted from website URLs and metadata. Each feature is encoded as:
- **-1** = Phishing indicator
- **0** = Suspicious / neutral
- **1** = Legitimate indicator

The target variable `Result` is: **1 = Phishing**, **-1 = Legitimate**.

In [ ]:
# Load dataset
df = pd.read_csv('data/phishingData.csv')
df.columns = df.columns.str.lower()

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names ({len(df.columns)}):")
print(list(df.columns))
print(f"\nData types:")
print(df.dtypes.value_counts())
df.head()

In [ ]:
# Detailed info
df.info()

In [ ]:
# Statistical summary
df.describe()

## 2. Thorough Exploratory Data Analysis

We perform **all** required data quality checks systematically.

### 2.1 Missing Values

In [ ]:
missing = df.isnull().sum()
print(f"Total missing values: {missing.sum()}")
if missing.sum() > 0:
    print(f"Columns with missing values:\n{missing[missing > 0]}")
else:
    print("No missing values found in any column.")

### 2.2 Infinite Values

In [ ]:
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f"Total infinite values: {inf_count}")

### 2.3 Constant Features

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Constant features (nunique <= 1): {constant_cols if constant_cols else 'None found'}")

### 2.4 Single-Value-Dominant Features (>95% one value)

In [ ]:
dominant_features = []
for col in df.columns:
    top_pct = df[col].value_counts(normalize=True).iloc[0]
    if top_pct > 0.95:
        dominant_features.append((col, top_pct, df[col].value_counts().index[0]))
        
if dominant_features:
    for col, pct, val in dominant_features:
        print(f"  {col}: {pct:.1%} is value {val}")
else:
    print("No single-value-dominant features found (all features have reasonable variation).")

### 2.5 Unique Values Per Column

In [ ]:
unique_info = pd.DataFrame({
    'column': df.columns,
    'nunique': [df[col].nunique() for col in df.columns],
    'unique_values': [sorted(df[col].unique()) for col in df.columns],
    'dtype': [df[col].dtype for col in df.columns]
})
unique_info

### 2.6 Exact Duplicate Rows

In [ ]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")

### 2.7 Conflicting Labels (Same Features, Different Target)
This checks for rows that have identical feature values but different `result` labels — a sign of label noise or ambiguity.

In [ ]:
feature_cols = [c for c in df.columns if c != 'result']
dup_features = df[df.duplicated(subset=feature_cols, keep=False)]
conflicting = dup_features.groupby(feature_cols)['result'].nunique()
conflicting_count = (conflicting > 1).sum()
print(f"Feature vectors with conflicting labels: {conflicting_count}")
if conflicting_count > 0:
    print("WARNING: Some identical feature vectors have different labels!")

### 2.8 Class Distribution

In [ ]:
class_dist = df['result'].value_counts()
imbalance_ratio = class_dist.max() / class_dist.min()

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Legitimate (-1)', 'Phishing (1)'],
              [class_dist.get(-1, 0), class_dist.get(1, 0)],
              color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, [class_dist.get(-1, 0), class_dist.get(1, 0)]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Class Distribution: Legitimate vs Phishing')
ax.set_ylim(0, max(class_dist) * 1.15)
plt.tight_layout()
plt.savefig('images/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClass counts: {class_dist.to_dict()}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")
print(f"The dataset is {'well balanced' if imbalance_ratio < 1.5 else 'imbalanced'}.")

### 2.9 Feature Distributions by Class

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(20, 20))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    ax = axes[i]
    for label, color in [(-1, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['result'] == label][col]
        ax.hist(subset, bins=5, alpha=0.6, color=color, label=f'{"Legit" if label==-1 else "Phish"}', edgecolor='black')
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=6)
    if i == 0:
        ax.legend(fontsize=6)
plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('images/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.10 Outlier Analysis (IQR Method)
Since features are ordinal (-1, 0, 1), traditional IQR outlier detection may not flag outliers. We verify this.

In [ ]:
outlier_summary = {}
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    if outliers > 0:
        outlier_summary[col] = outliers

if outlier_summary:
    print("Columns with outliers (IQR method):")
    for col, count in outlier_summary.items():
        print(f"  {col}: {count} outliers ({count/len(df)*100:.2f}%)")
else:
    print("No traditional outliers detected.")
    print("This is expected because all features are ordinal (-1, 0, 1) — IQR method is less meaningful here.")

### 2.11 Correlation Analysis (Spearman)
We use Spearman correlation because our features are ordinal, not continuous.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))
corr_spearman = df.corr(method='spearman')
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
sns.heatmap(corr_spearman, mask=mask, annot=False, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Spearman Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/03_correlation_spearman.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with target
target_corr = corr_spearman['result'].drop('result').abs().sort_values(ascending=False)
print("Top 10 features most correlated with target (|Spearman r|):")
for feat, val in target_corr.head(10).items():
    direction = "+" if corr_spearman.loc[feat, 'result'] > 0 else "-"
    print(f"  {feat}: {direction}{val:.4f}")

### 2.12 Redundancy Analysis (Highly Correlated Feature Pairs)

In [ ]:
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        c = abs(corr_spearman.loc[feature_cols[i], feature_cols[j]])
        if c > 0.5:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], round(c, 4)))
high_corr_pairs.sort(key=lambda x: x[2], reverse=True)

if high_corr_pairs:
    print(f"Feature pairs with |Spearman r| > 0.5 (potential redundancy):")
    for f1, f2, c in high_corr_pairs:
        print(f"  {f1} <-> {f2}: {c}")
else:
    print("No highly correlated feature pairs found.")

### 2.13 Group-By Analysis (Feature Means by Class)

In [ ]:
group_means = df.groupby('result')[feature_cols].mean()
diff = (group_means.loc[1] - group_means.loc[-1]).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(diff))
ax.bar(x, diff.values, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(diff.index, rotation=90, fontsize=7)
ax.set_ylabel('|Mean(Phishing) - Mean(Legitimate)|')
ax.set_title('Absolute Difference in Feature Means Between Classes', fontweight='bold')
plt.tight_layout()
plt.savefig('images/04_groupby_mean_diff.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 most discriminative features (by mean difference):")
for feat, val in diff.head(10).items():
    print(f"  {feat}: {val:.4f}")

### 2.14 Crosstab Analysis (Top 6 Discriminative Features)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
top_features = diff.head(6).index.tolist()
for idx, feat in enumerate(top_features):
    ax = axes[idx // 3][idx % 3]
    ct = pd.crosstab(df[feat], df['result'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{feat} vs Result', fontsize=10)
    ax.set_ylabel('Proportion')
    ax.legend(['Legit (-1)', 'Phish (1)'], fontsize=7)
    ax.tick_params(labelsize=8)
plt.suptitle('Top 6 Discriminative Features: Crosstab Analysis', fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('images/05_crosstab_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.15 Drop Duplicates & Prepare for Modeling

In [ ]:
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

## 3. Data Preparation (Leakage-Safe Splitting)

We use a **stratified 60/20/20 split** into train, validation, and test sets. The test set remains untouched until final evaluation.

In [ ]:
X = df.drop(columns=['result'])
y = df['result'].map({-1: 0, 1: 1})  # Map to standard binary: 0=Legitimate, 1=Phishing

# Stratified 60/20/20 split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")
print(f"Val class dist:   {y_val.value_counts().to_dict()}")
print(f"Test class dist:  {y_test.value_counts().to_dict()}")

# Leakage check
train_idx, val_idx, test_idx = set(X_train.index), set(X_val.index), set(X_test.index)
assert len(train_idx & val_idx) == 0 and len(train_idx & test_idx) == 0 and len(val_idx & test_idx) == 0
print("\nLeakage check PASSED: No overlap between train/val/test sets.")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 4. Feature Engineering Experiments

We analyze which features are most important and test whether reducing the feature set improves or hurts performance.

### 4.1 Random Forest Feature Importance

In [ ]:
rf_imp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imp.fit(X_train, y_train)
importances = pd.Series(rf_imp.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Random Forest Feature Importance', fontweight='bold')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('images/06_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 most important features:")
for feat, val in importances.head(10).items():
    print(f"  {feat}: {val:.4f}")

### 4.2 Mutual Information Scores

In [ ]:
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
mi_series.plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_title('Mutual Information Scores', fontweight='bold')
ax.set_ylabel('MI Score')
plt.tight_layout()
plt.savefig('images/07_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 features by mutual information:")
for feat, val in mi_series.head(10).items():
    print(f"  {feat}: {val:.4f}")

### 4.3 Recursive Feature Elimination (RFE)

In [ ]:
rfe_model = RandomForestClassifier(n_estimators=50, random_state=42)
rfe = RFE(rfe_model, n_features_to_select=15, step=1)
rfe.fit(X_train, y_train)
rfe_selected = X_train.columns[rfe.support_].tolist()
print(f"RFE selected features (top 15):\n{rfe_selected}")

### 4.4 PCA Analysis

In [ ]:
pca = PCA(random_state=42)
pca.fit(X_train_scaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar)+1), cumvar, 'bo-', markersize=4)
ax.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
ax.axhline(y=0.90, color='orange', linestyle='--', label='90% variance')
n_95 = np.argmax(cumvar >= 0.95) + 1
n_90 = np.argmax(cumvar >= 0.90) + 1
ax.axvline(x=n_95, color='r', linestyle=':', alpha=0.5)
ax.axvline(x=n_90, color='orange', linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA: Cumulative Explained Variance', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('images/08_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")

### 4.5 Feature Set Comparison
We compare the Random Forest accuracy using different subsets of features to measure the impact of feature selection.

In [ ]:
top_15 = importances.head(15).index.tolist()
top_10 = importances.head(10).index.tolist()

feature_sets = {
    'All 30 features': X_train.columns.tolist(),
    'Top 15 (RF importance)': top_15,
    'Top 10 (RF importance)': top_10,
    'RFE 15 features': rfe_selected,
}

feat_comparison = {}
for name, feats in feature_sets.items():
    rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
    scores = cross_val_score(rf_temp, X_train[feats], y_train, cv=5, scoring='accuracy')
    feat_comparison[name] = {'mean_acc': scores.mean(), 'std': scores.std()}
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
names = list(feat_comparison.keys())
means = [feat_comparison[n]['mean_acc'] for n in names]
stds = [feat_comparison[n]['std'] for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=['#3498db', '#2ecc71', '#e67e22', '#9b59b6'], edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Cross-Validation Accuracy')
ax.set_title('Feature Set Comparison (Random Forest, 5-Fold CV)', fontweight='bold')
ax.set_ylim(min(means) - 0.02, max(means) + 0.015)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('images/09_feature_set_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training and Comparison

We train **5 different classifiers** and compare them using 5-fold cross-validation on the training set. Models requiring feature scaling (LR, SVM, KNN) use the scaled data.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    print(f"\n--- Training: {name} ---")
    if name in ['Logistic Regression', 'SVM (RBF)', 'KNN']:
        X_tr, X_v = X_train_scaled, X_val_scaled
    else:
        X_tr, X_v = X_train.values, X_val.values

    # Cross-validation
    scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')
    print(f"  CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

    # Fit and validate
    model.fit(X_tr, y_train)
    val_preds = model.predict(X_v)
    val_probs = model.predict_proba(X_v)[:, 1]
    val_acc = accuracy_score(y_val, val_preds)
    print(f"  Validation Accuracy: {val_acc:.4f}")

    cv_results[name] = {
        'model': model, 'cv_mean': scores.mean(), 'cv_std': scores.std(),
        'val_acc': val_acc, 'val_preds': val_preds, 'val_probs': val_probs,
    }

In [ ]:
# Cross-validation comparison
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(cv_results.keys())
cv_means = [cv_results[n]['cv_mean'] for n in model_names]
cv_stds = [cv_results[n]['cv_std'] for n in model_names]
colors_m = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#e74c3c']
bars = ax.bar(model_names, cv_means, yerr=cv_stds, capsize=5, color=colors_m, edgecolor='black')
for bar, val in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Model Comparison: Cross-Validation Accuracy', fontweight='bold')
ax.set_ylim(min(cv_means) - 0.03, max(cv_means) + 0.02)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('images/10_model_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Final Evaluation on Test Set

We evaluate all 5 models on the **untouched test set** using comprehensive metrics:
Accuracy, Precision, Recall, F1, F2, MCC, ROC-AUC, PR-AUC, Confusion Matrix, FPR, FNR.

In [ ]:
all_metrics = {}

for name in model_names:
    model = cv_results[name]['model']
    X_te = X_test_scaled if name in ['Logistic Regression', 'SVM (RBF)', 'KNN'] else X_test.values

    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1]

    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()

    all_metrics[name] = {
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1': round(f1_score(y_test, preds), 4),
        'F2': round(fbeta_score(y_test, preds, beta=2), 4),
        'MCC': round(matthews_corrcoef(y_test, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 4),
        'PR-AUC': round(average_precision_score(y_test, probs), 4),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'FPR': round(fp / (fp + tn), 4),
        'FNR': round(fn / (fn + tp), 4),
    }

# Display as table
metrics_df = pd.DataFrame(all_metrics).T
display(metrics_df)

### Confusion Matrices — All Models

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
for idx, name in enumerate(model_names):
    m = all_metrics[name]
    cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit (0)', 'Phish (1)'], yticklabels=['Legit (0)', 'Phish (1)'])
    ax.set_title(f'{name}\nAcc={m["Accuracy"]:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('images/11_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### ROC and Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for name in model_names:
    model = cv_results[name]['model']
    X_te = X_test_scaled if name in ['Logistic Regression', 'SVM (RBF)', 'KNN'] else X_test.values
    probs = model.predict_proba(X_te)[:, 1]

    fpr_c, tpr_c, _ = roc_curve(y_test, probs)
    prec_c, rec_c, _ = precision_recall_curve(y_test, probs)

    ax1.plot(fpr_c, tpr_c, label=f'{name} (AUC={all_metrics[name]["ROC-AUC"]:.3f})')
    ax2.plot(rec_c, prec_c, label=f'{name} (AP={all_metrics[name]["PR-AUC"]:.3f})')

ax1.plot([0,1], [0,1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves — All Models', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves — All Models', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/12_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Selection

In phishing detection, **False Negatives are more dangerous than False Positives** — a phishing site slipping through can cause data theft. We optimize the threshold using the F2 score (which weights recall higher) on the validation set.

In [ ]:
best_model = cv_results['Random Forest']['model']
val_probs_best = cv_results['Random Forest']['val_probs']

thresholds = np.arange(0.1, 0.91, 0.05)
thresh_results = []
for t in thresholds:
    preds_t = (val_probs_best >= t).astype(int)
    thresh_results.append({
        'threshold': t,
        'precision': precision_score(y_val, preds_t, zero_division=0),
        'recall': recall_score(y_val, preds_t, zero_division=0),
        'f1': f1_score(y_val, preds_t, zero_division=0),
        'f2': fbeta_score(y_val, preds_t, beta=2, zero_division=0)
    })

thresh_df = pd.DataFrame(thresh_results)
optimal_threshold = thresh_df.loc[thresh_df['f2'].idxmax(), 'threshold']
print(f"Optimal threshold (max F2 on validation): {optimal_threshold:.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['threshold'], thresh_df['precision'], 'b-o', ms=4, label='Precision')
ax.plot(thresh_df['threshold'], thresh_df['recall'], 'r-s', ms=4, label='Recall')
ax.plot(thresh_df['threshold'], thresh_df['f1'], 'g-^', ms=4, label='F1')
ax.plot(thresh_df['threshold'], thresh_df['f2'], 'm-D', ms=4, label='F2')
ax.axvline(x=optimal_threshold, color='black', linestyle='--', label=f'Optimal={optimal_threshold:.2f}')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Selection on Validation Set (Random Forest)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/13_threshold_selection.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare on test set
test_probs = best_model.predict_proba(X_test.values)[:, 1]
default_preds = best_model.predict(X_test.values)
tuned_preds = (test_probs >= optimal_threshold).astype(int)

print(f"\nTest Set Comparison:")
print(f"  Default (0.5): Recall={recall_score(y_test, default_preds):.4f}, F1={f1_score(y_test, default_preds):.4f}")
print(f"  Tuned ({optimal_threshold:.2f}):  Recall={recall_score(y_test, tuned_preds):.4f}, F1={f1_score(y_test, tuned_preds):.4f}")

## 8. Error Analysis: False Positives and False Negatives

In cybersecurity, errors have different costs:
- **False Negative (FN)**: A phishing website is classified as legitimate → **User data theft, financial loss** (HIGH COST)
- **False Positive (FP)**: A legitimate website is blocked → **User inconvenience** (LOWER COST)

We inspect the actual misclassified examples to understand failure patterns.

In [ ]:
# Build error analysis DataFrame
errors_df = X_test.copy()
errors_df['actual'] = y_test.values
errors_df['predicted'] = default_preds
errors_df['prob_phishing'] = test_probs

fn_df = errors_df[(errors_df['actual'] == 1) & (errors_df['predicted'] == 0)]
fp_df = errors_df[(errors_df['actual'] == 0) & (errors_df['predicted'] == 1)]
correct_phish = errors_df[(errors_df['actual'] == 1) & (errors_df['predicted'] == 1)]
correct_legit = errors_df[(errors_df['actual'] == 0) & (errors_df['predicted'] == 0)]

print(f"Total test samples: {len(errors_df)}")
print(f"False Negatives (Phishing MISSED):   {len(fn_df)}")
print(f"False Positives (Legitimate BLOCKED): {len(fp_df)}")
print(f"Correctly classified:                 {len(correct_phish) + len(correct_legit)}")

### 8.1 False Negative Analysis
Which features cause phishing sites to be misclassified as legitimate?

In [ ]:
if len(fn_df) > 0 and len(correct_phish) > 0:
    fn_means = fn_df[feature_cols].mean()
    cp_means = correct_phish[feature_cols].mean()
    diff_fn = (fn_means - cp_means).abs().sort_values(ascending=False)
    
    print("Features where False Negatives differ most from correctly classified phishing:")
    for feat, val in diff_fn.head(10).items():
        print(f"  {feat}: |diff|={val:.4f} (FN mean={fn_means[feat]:.3f}, correct phish mean={cp_means[feat]:.3f})")
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for idx, feat in enumerate(diff_fn.head(6).index):
        ax = axes[idx // 3][idx % 3]
        data_plot = pd.DataFrame({
            'Correct Phishing': correct_phish[feat].value_counts(normalize=True),
            'False Negatives': fn_df[feat].value_counts(normalize=True)
        }).fillna(0)
        data_plot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
        ax.set_title(f'{feat}', fontsize=10)
        ax.set_ylabel('Proportion')
    plt.suptitle('False Negatives vs Correctly Classified Phishing\n(Feature Distribution Comparison)', fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig('images/14_fn_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No false negatives to analyze.")

### 8.2 False Positive Analysis
Which features cause legitimate sites to be misclassified as phishing?

In [ ]:
if len(fp_df) > 0 and len(correct_legit) > 0:
    fp_means = fp_df[feature_cols].mean()
    cl_means = correct_legit[feature_cols].mean()
    diff_fp = (fp_means - cl_means).abs().sort_values(ascending=False)
    
    print("Features where False Positives differ most from correctly classified legitimate:")
    for feat, val in diff_fp.head(10).items():
        print(f"  {feat}: |diff|={val:.4f} (FP mean={fp_means[feat]:.3f}, correct legit mean={cl_means[feat]:.3f})")
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for idx, feat in enumerate(diff_fp.head(6).index):
        ax = axes[idx // 3][idx % 3]
        data_plot = pd.DataFrame({
            'Correct Legitimate': correct_legit[feat].value_counts(normalize=True),
            'False Positives': fp_df[feat].value_counts(normalize=True)
        }).fillna(0)
        data_plot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
        ax.set_title(f'{feat}', fontsize=10)
        ax.set_ylabel('Proportion')
    plt.suptitle('False Positives vs Correctly Classified Legitimate\n(Feature Distribution Comparison)', fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig('images/15_fp_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No false positives to analyze.")

### 8.3 Sample Misclassified Examples

In [ ]:
print("Sample FALSE NEGATIVE examples (Phishing sites classified as Legitimate):")
if len(fn_df) > 0:
    display(fn_df.head())
    
print("\nSample FALSE POSITIVE examples (Legitimate sites classified as Phishing):")
if len(fp_df) > 0:
    display(fp_df.head())

## 9. Faithful Reproduction of Author's Original Experiment

The original author ([sujeetgund/phishing-website-detection](https://github.com/sujeetgund/phishing-website-detection)) claims the following results using 5-fold cross-validation:

| Model | Author's Mean Test Score | Author's Std |
|---|---|---|
| RandomForest | 0.9711 | 0.0041 |
| SVC | 0.9629 | 0.0064 |
| KNeighbors | 0.9623 | 0.0046 |
| Logistic | 0.9270 | 0.0047 |
| Ridge | 0.9206 | 0.0053 |

We now reproduce this experiment as faithfully as possible using the same models and default hyperparameters on the **original, unmodified dataset**.

In [ ]:
# Reload original data (no duplicate removal, no target remapping)
df_orig = pd.read_csv('data/phishingData.csv')
df_orig.columns = df_orig.columns.str.lower()
X_orig = df_orig.drop(columns=['result'])
y_orig = df_orig['result']

author_models = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'KNeighbors': KNeighborsClassifier(),
    'Logistic': LogisticRegression(max_iter=1000, random_state=42),
    'Ridge': RidgeClassifier(random_state=42),
}

author_reported = {
    'RandomForest': {'mean': 0.9711, 'std': 0.0041},
    'SVC':          {'mean': 0.9629, 'std': 0.0064},
    'KNeighbors':   {'mean': 0.9623, 'std': 0.0046},
    'Logistic':     {'mean': 0.9270, 'std': 0.0047},
    'Ridge':        {'mean': 0.9206, 'std': 0.0053},
}

reproduction_results = {}
for name, model in author_models.items():
    if name in ['SVC', 'Logistic', 'KNeighbors']:
        X_cv = StandardScaler().fit_transform(X_orig)
    else:
        X_cv = X_orig.values

    scores = cross_val_score(model, X_cv, y_orig, cv=5, scoring='accuracy')
    delta = abs(scores.mean() - author_reported[name]['mean'])
    reproduction_results[name] = {
        'our_mean': round(scores.mean(), 4), 'our_std': round(scores.std(), 4),
        'author_mean': author_reported[name]['mean'], 'author_std': author_reported[name]['std'],
        'delta': round(delta, 4)
    }
    print(f"{name}: Author={author_reported[name]['mean']:.4f}, Ours={scores.mean():.4f}, Delta={delta:.4f}")

In [ ]:
# Reproduction comparison chart
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(reproduction_results))
width = 0.35
a_means = [reproduction_results[n]['author_mean'] for n in reproduction_results]
o_means = [reproduction_results[n]['our_mean'] for n in reproduction_results]
a_stds = [reproduction_results[n]['author_std'] for n in reproduction_results]
o_stds = [reproduction_results[n]['our_std'] for n in reproduction_results]

ax.bar(x_pos - width/2, a_means, width, yerr=a_stds, capsize=4, label="Author's Reported", color='#3498db', edgecolor='black', alpha=0.8)
ax.bar(x_pos + width/2, o_means, width, yerr=o_stds, capsize=4, label="Our Reproduction", color='#e67e22', edgecolor='black', alpha=0.8)
ax.set_ylabel('Mean CV Accuracy')
ax.set_title("Author's Reported vs Our Reproduced Results", fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(list(reproduction_results.keys()))
ax.legend(); ax.set_ylim(0.90, 1.0); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/16_reproduction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
repro_df = pd.DataFrame(reproduction_results).T
display(repro_df)

## 10. Conclusions

**Key findings from this reproduction study:**

1. **Author's claims are verified**: Our reproduced Random Forest results closely match the reported ~97.1% accuracy.
2. **Random Forest is the best model** for this dataset, followed by Gradient Boosting, SVM, and KNN. Logistic Regression performs the weakest.
3. **Feature engineering impact**: Using 15 features selected by RFE achieves nearly the same accuracy as using all 30, suggesting some features are redundant.
4. **Error analysis**: False Negatives are driven by specific features (e.g., SSL state, URL characteristics) that make some phishing sites appear legitimate.
5. **Threshold tuning**: Lowering the decision threshold improves recall at the cost of precision — a worthwhile trade-off in phishing detection.

**Limitations:**
- The dataset uses pre-extracted features from 2015; modern phishing tactics may not be captured.
- All features are ordinal (-1, 0, 1), limiting the value of continuous-data techniques like PCA.
- No temporal analysis is possible as the dataset lacks timestamps.
